Difficulty Levels

🐢 Use the `HuggingFaceTB/smoltalk` dataset

🐕 Try out the `bigcode/the-stack-smol` dataset and finetune a code generation model on a specific subset `data/python`.

🦁 Select a dataset that relates to a real world use case your interested in


In [ ]:
from huggingface_hub import login
login()

In [ ]:
%pip install trl==0.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 10.7 MB/s eta 0:00:00


In [ ]:
%pip show trl

Name: trl
Version: 0.12.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: Leandro von Werra
Author-email: leandro.vonwerra@gmail.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, datasets, rich, transformers
Required-by: 


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch

device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path="HuggingfaceTB/SmolLM2-135M"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="HuggingfaceTB/SmolLM2-135M")
# since we disable chat template for code fullfilling task
# we need to set pad_token due to STTTrain need
tokenizer.pad_token = tokenizer.eos_token
# setup for chat mode in default ChatML template
# notice: use tokenizer.apply_chat_template() for instruct model && tokenizer.chat_template() for self-defined chat
# instruct model: product of base model being instruct fine-tuned
# model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

# comment out the last line and add the below line in code filling task
tokenizer.chat_template = None
# this will ban conversation template

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

In [ ]:
# test our base model
prompt = "def quicksort(arr):"

# message = [{"role": "user", "content": prompt}]
# formatted_prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt').to(device)
outputs = model.generate(**inputs,
  max_new_tokens=300,
  temperature=0.8,
  top_p=0.9,
  repetition_penalty=1.2,
  do_sample=True,
  eos_token_id=tokenizer.eos_token_id,
)
print(f"Base model outputs: (prompt: {prompt})")
# or
# print("Base model outputs: (prompt: {})".format(prompt))
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Base model outputs: (prompt: def quicksort(arr):)
def quicksort(arr):
    if len(arr) == 1:
        return arr[0]

    result = arr[:len(arr)-2]; 
    qq_sort([x.lower() for x in range(len(result))], [c]); 

# Quicksorts the first element of an array, which is less than or equal to a given number n (the sort order). This can be useful when sorting arrays with very different lengths but similar data types, and also helps reduce memory usage due to fast comparisons between elements sorted at each end as compared to comparing them one by one on-the-fly using lookup tables.
"""
  def _quicksortedArrayQuotientSort(self,a=[]):
      # Quickly partition from left into halves
      
          while not self._isComplete():
              tt=min((i+j)+3)/4
                  for i,(k , j )in enumerate(zip(*list(_partition())) ):
                      v=(v%8)*int(''.join([[str(ii),s]+['{', '}']*(tt - ii + 1)))
                              s=' '.join(['{} {} {}'.format(', %d'%(n,' ', str(iii)), '%'

In [ ]:
# Data Preparation
ds = load_dataset("bigcode/the-stack-smol", data_dir="data/python", split='train')
ds

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

data/python/data.json:   0%|          | 0.00/87.0M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['content', 'avg_line_length', 'max_line_length', 'alphanum_fraction', 'licenses', 'repository_name', 'path', 'size', 'lang'],
    num_rows: 10000
})

In [ ]:
ds = ds.train_test_split(test_size=0.1)

In [ ]:
ds['train'][0]['content']

'import pandas as pd\nimport plotly.express as px\n\ndf = pd.read_csv(\'data/query_result.csv\')\nmax_df = df.groupby(by=\'topic_id\').max().reset_index()\ndf = df[df[\'topic_id\'].isin(max_df[max_df[\'recall\'] > 0][\'topic_id\'].to_list())]\nfor t in df[\'topic_id\'].unique().tolist():\n    temp_df = df[df[\'topic_id\'] == t]\n    fig = px.box(df, x="topic_id", y="recall")\n    fig.update_traces(quartilemethod="exclusive")  # or "inclusive", or "linear" by default\n    fig.show()\n'

In [ ]:
def formatting_func(raw_data):
  # model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)
  return raw_data["content"]

In [ ]:
# Configure SFTTrainer
sft_config = SFTConfig(
    output_dir='./SFT_output',
    max_steps=200,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10, # this will print logs every 10 steps(loss, lr...)
    save_steps=100,
    eval_strategy='steps',
    eval_steps=50,
    hub_model_id="SFT_on_SmolTalk",
  )

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    tokenizer=tokenizer,
    eval_dataset=ds["test"],
    formatting_func=formatting_func
)

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
trainer.save_model(f"./SFT_on_SmolTalk")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss,Validation Loss
50,1.463600,1.469297
100,1.404900,1.461073
150,1.290700,1.456363
200,1.395500,1.454780


In [ ]:
# test our fine-tuned model
prompt = "def quicksort(arr):"

# message = [{"role": "user", "content": prompt}]
# formatted_prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt').to(device)
outputs_post = model.generate(**inputs,
  max_new_tokens=300,
  temperature=0.8,
  top_p=0.9,
  repetition_penalty=1.2,
  do_sample=True,
  eos_token_id=tokenizer.eos_token_id,
)
print(f"Fine-tuned model outputs: (prompt: {prompt})")
# or
# print("Base model outputs: (prompt: {})".format(prompt))
print(tokenizer.decode(outputs_post[0], skip_special_tokens=True))
print("Base model outputs: (prompt: {})".format(prompt))
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Fine-tuned model outputs: (prompt: def quicksort(arr):)
def quicksort(arr):
    sorted_list = []

    for num in range (1, len(arr)):
        tempnum= arr[0]

        leftmost=(tempnum - 2) % n + 1
        
        rightleast=[]    
    
        while(leftmost<n-len() and rightlest==None or ((rightest+lengen)<i.size())and((xor < i[-3:]))) :
            xofirst=[xo]*maxlength
            
            if xonight>nums:
                yojnd=-y;

                
                
                if not all([k > l else k >= a]:
                    #print("The last element of the array is already greater than this one", *sorted).center(' ', maxindex )
                    

                    tmpsorted=[''].append([])
                        tempsums=[],tmpkind='pre'.format(*sorted),
                            for tupolist in [a,-b:-jardom].zipWithIndex():
                                tmppartstargeting['']+=troup[:]
                                   
                            
       

In [ ]:
# packing allow multiple short examples packed into sequence of the same length
# which can maximine GPU utilization but may bring along more epochs
training_args = SFTConfig(packing=True)

Warning Signs during Training:<br>-Validation loss increasing while training loss decreases (overfitting)<br>-No significant improvement in loss values (underfitting)<br>-Extremely low loss values (potential memorization)<br>-Inconsistent output formatting (template learning issues)